# Demo Media Capture

Rekam demo Gradio nyata untuk aset rilis. Gunakan runtime baru, pilih **Runtime > Change runtime type > T4 GPU**, lalu jalankan **Run all**. Notebook akan mengunduh satu ZIP berisi video WebM, screenshot flow berhasil dan abstain, log startup, serta manifest checksum.

In [ ]:
import subprocess

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).strip()
assert gpu, "Pilih Runtime > Change runtime type > T4 GPU"
print(gpu)

In [ ]:
import os
import subprocess
from pathlib import Path

BRANCH = "main"
repo_dir = Path("/content/indonesian-legal-compliance-rag")
if not (repo_dir / ".git").is_dir():
    subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", BRANCH,
            "https://github.com/FadhilahAfif/indonesian-legal-compliance-rag.git",
            str(repo_dir),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "checkout", BRANCH], cwd=repo_dir, check=True)
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=repo_dir,
        check=True,
    )
os.chdir(repo_dir)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "playwright==1.61.0"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "uninstall", "-y", "-q",
        "torchvision", "torchcodec", "torchaudio",
    ],
    check=True,
)
subprocess.run([sys.executable, "scripts/check_environment.py"], check=True)
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    check=True,
)

In [ ]:
import subprocess
import sys
from pathlib import Path

from src.rag import REGULATION_BY_FILE

subprocess.run(
    [
        sys.executable, "-m", "gdown", "--folder",
        "https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql",
        "-O", "data/raw",
    ],
    check=True,
)
corpus_dir = Path("data/raw")
missing = [name for name in REGULATION_BY_FILE if not (corpus_dir / name).is_file()]
assert not missing, f"PDF corpus tidak lengkap: {missing}"
print(f"Corpus OK: {len(REGULATION_BY_FILE)} PDF")

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

APP_URL = "http://127.0.0.1:7860"
log_path = Path("/content/demo_app.log")
log_handle = log_path.open("w", encoding="utf-8")
app_process = subprocess.Popen(
    [sys.executable, "app.py", "--device", "cuda"],
    cwd=repo_dir,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    env={**os.environ, "GRADIO_ANALYTICS_ENABLED": "False"},
)
for _ in range(900):
    if app_process.poll() is not None:
        log_handle.flush()
        raise RuntimeError(log_path.read_text(encoding="utf-8")[-4000:])
    try:
        with urlopen(APP_URL, timeout=2) as response:
            if response.status == 200:
                break
    except (URLError, TimeoutError, ConnectionError):
        time.sleep(1)
else:
    app_process.terminate()
    raise TimeoutError("Demo tidak siap dalam 15 menit; periksa demo_app.log")
print(f"Demo siap: {APP_URL}")

In [ ]:
import hashlib
import json
import platform
import re
import shutil
import subprocess
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import torch
from playwright.sync_api import sync_playwright

answerable_question = "Saya pekerja harian yang bekerja 22 hari setiap bulan selama tiga bulan berturut-turut. Apakah perjanjian kerja harian saya masih berlaku?"
unanswerable_question = "Berapa nominal UMK Kota Bandung yang berlaku saat ini?"
captured_at = datetime.now(timezone.utc)
capture_dir = Path("/content") / captured_at.strftime("demo-media-%Y%m%dT%H%M%SZ")
video_dir = capture_dir / "raw-video"
video_dir.mkdir(parents=True)

with sync_playwright() as playwright:
    browser = playwright.chromium.launch(headless=True)
    context = browser.new_context(
        viewport={"width": 1440, "height": 1000},
        record_video_dir=str(video_dir),
        record_video_size={"width": 1440, "height": 1000},
    )
    page = context.new_page()
    page.goto(APP_URL, wait_until="domcontentloaded", timeout=120_000)
    page.get_by_role("heading", name="Asisten Kepatuhan Hukum Indonesia").wait_for(
        timeout=120_000
    )
    page.wait_for_timeout(1500)

    question = page.get_by_label("Pertanyaan")
    question.press_sequentially(answerable_question, delay=25)
    page.get_by_role("button", name="Kirim").click()
    page.get_by_text(re.compile(r"Status:\s*Berhasil - bukti ditemukan")).wait_for(
        timeout=180_000
    )
    page.wait_for_timeout(2500)
    page.screenshot(path=str(capture_dir / "answerable.png"), full_page=True)

    page.get_by_role("button", name="Bersihkan").click()
    page.get_by_text(re.compile(r"Status:\s*Siap")).wait_for(timeout=30_000)
    question.press_sequentially(unanswerable_question, delay=25)
    page.get_by_role("button", name="Kirim").click()
    page.get_by_text(re.compile(r"Status:\s*Konteks tidak cukup")).wait_for(
        timeout=180_000
    )
    page.wait_for_timeout(2500)
    page.screenshot(path=str(capture_dir / "unanswerable.png"), full_page=True)

    context.close()
    recorded_video = Path(page.video.path())
    browser.close()

shutil.move(str(recorded_video), capture_dir / "demo.webm")
app_process.terminate()
try:
    app_process.wait(timeout=30)
except subprocess.TimeoutExpired:
    app_process.kill()
    app_process.wait()
log_handle.close()
shutil.copy2(log_path, capture_dir / "app.log")
shutil.rmtree(video_dir)

media_files = [capture_dir / name for name in ("demo.webm", "answerable.png", "unanswerable.png")]
manifest = {
    "captured_at_utc": captured_at.isoformat(),
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True
    ).strip(),
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "playwright": version("playwright"),
    },
    "viewport": {"width": 1440, "height": 1000},
    "flows": [
        {"name": "answerable", "question": answerable_question},
        {"name": "unanswerable", "question": unanswerable_question},
    ],
    "files": {
        path.name: {
            "bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
        for path in media_files
    },
}
(capture_dir / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
archive_path = Path(shutil.make_archive(str(capture_dir), "zip", root_dir=capture_dir))
print(json.dumps({"archive": str(archive_path), **manifest}, indent=2))

In [ ]:
from google.colab import files

files.download(str(archive_path))